# ConsistEdit → Modular Diffusers — E2E (training-free editing, adjustable structural consistency)

Masked text editing on FLUX with pre-attention vision-token fusion (ConsistEdit, arXiv:2510.17803). Publish PRIVATE `remyxai/consistedit-flux-modular` → load via `trust_remote_code` → real edits → **quantitative validation**: background SSIM/L1 on the kept region + Canny SSIM (structure) + CLIP edit-direction, a `consistency_strength` sweep (the paper's headline capability), and a head-to-head vs FlowEdit and KV-Edit. Upload `block.py` first. Runtime: A100 · `HUGGINGFACE_TOKEN` · accept FLUX.1-dev.

## 1 · Install + GPU + auth

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf scikit-image

In [ ]:
import torch
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16


## 2 · Publish PRIVATE (upload block.py first)

In [ ]:
import os, json
from huggingface_hub import HfApi
assert os.path.exists("block.py"), "Upload block.py first."
open("modular_config.json","w").write(json.dumps({"_class_name":"ConsistEditBlock","_diffusers_version":"0.41.0.dev0","auto_map":{"ModularPipelineBlocks":"block.ConsistEditBlock"}},indent=2))
F="black-forest-labs/FLUX.1-dev"
def c(s,l,cl): return [None,None,{"pretrained_model_name_or_path":F,"revision":None,"subfolder":s,"type_hint":[l,cl],"variant":None}]
open("modular_model_index.json","w").write(json.dumps({"_blocks_class_name":"ConsistEditBlock","_class_name":"ModularPipeline","_diffusers_version":"0.41.0.dev0",
 "text_encoder":c("text_encoder","transformers","CLIPTextModel"),"tokenizer":c("tokenizer","transformers","CLIPTokenizer"),
 "text_encoder_2":c("text_encoder_2","transformers","T5EncoderModel"),"tokenizer_2":c("tokenizer_2","transformers","T5TokenizerFast"),
 "transformer":c("transformer","diffusers","FluxTransformer2DModel"),"vae":c("vae","diffusers","AutoencoderKL"),
 "scheduler":c("scheduler","diffusers","FlowMatchEulerDiscreteScheduler")},indent=2))
api=HfApi(); REPO="remyxai/consistedit-flux-modular"; api.create_repo(REPO,private=True,repo_type="model",exist_ok=True)
for f in ["block.py","modular_config.json","modular_model_index.json"]: api.upload_file(path_or_fileobj=f,path_in_repo=f,repo_id=REPO)
print("published PRIVATE:", api.list_repo_files(REPO))


## 3 · Load + a source image + mask

In [ ]:
from diffusers import ModularPipeline
from PIL import Image, ImageDraw
from io import BytesIO
import requests
from IPython.display import display
pipe = ModularPipeline.from_pretrained("remyxai/consistedit-flux-modular", trust_remote_code=True)
assert type(pipe.blocks).__name__ == "ConsistEditBlock", type(pipe.blocks).__name__
print("loaded block:", type(pipe.blocks).__name__)   # expect ConsistEditBlock
pipe.load_components(dtype=DT); pipe.to(DEV)

# The source MUST be a cat so source_prompt "a cat" matches — ConsistEdit RF-inverts under the source
# prompt, and a mismatched prompt breaks the inversion (and the edit). Try reliable mirrors first.
SRC_URLS = ["https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/cat.png",
            "https://raw.githubusercontent.com/fallenshock/FlowEdit/main/inputs/cat.png"]
src = None
for u in SRC_URLS:
    try:
        src = Image.open(BytesIO(requests.get(u, timeout=30).content)).convert("RGB"); print("source:", u); break
    except Exception as e:
        print("fetch failed:", u, "->", e)
if src is None:   # last resort: upload a CAT (and keep source_prompt='a cat' below)
    from google.colab import files; up=files.upload(); src=Image.open(list(up.keys())[0]).convert("RGB")
src = src.resize((1024,1024)); src.save("src.png")

# edit mask OVER THE SUBJECT (white=edit, black=keep): a centred box covering the animal so the subject
# swap lands on it, while the border stays pixel-stable. Enlarged from the old 512 box to cover the cat.
mask = Image.new("L", (1024,1024), 0)
ImageDraw.Draw(mask).rectangle([192, 192, 832, 832], fill=255)
mask.save("mask.png")
print("source | mask:"); display(src.resize((320,320))); display(mask.resize((320,320)))


## 4 · Spike first — no-op when the feature is off

Run `smoke.ipynb` milestone B first if you haven't: `consistency_strength=0` must disarm the seam bit-exactly, and an all-keep mask must reconstruct the source. Those exercise the exact capture/fuse path this notebook relies on.

## 5 · The edits (full settings)

Both edit flavours the paper separates: a **texture edit** (same animal, different colour — should keep the pose/folds at α=1) and a **subject swap** (structure changes — α=1 should still hold the layout).

In [ ]:
import torch
from PIL import Image
from IPython.display import display
EDITS = [("a cat", "a siamese cat"), ("a cat", "a dog")]   # (source_prompt, target_prompt)
outs = [("source", src), ("mask", mask.convert("RGB"))]
for sp, tp in EDITS:
    g = torch.Generator(DEV).manual_seed(0)
    im = pipe(image="src.png", mask="mask.png", source_prompt=sp, prompt=tp,
              height=1024, width=1024, T_steps=28, guidance_scale=3.5, src_guidance_scale=1.0,
              consistency_strength=0.3, generator=g).images[0]
    im.save(f"edit_{tp.replace(' ','_')}.png"); outs.append((tp, im)); print("  ✓", tp)
S = 384; W_ = len(outs)*S + (len(outs)+1)*10
row = Image.new("RGB", (W_, S+40), "white")
from PIL import ImageDraw, ImageFont
d = ImageDraw.Draw(row)
try: Ft = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 26)
except Exception: Ft = ImageFont.load_default()
for i, (name, im) in enumerate(outs):
    x = 10 + i*(S+10); row.paste(im.resize((S,S)), (x,0)); d.text((x+8, S+6), name, fill="black", font=Ft)
row.save("consistedit_grid.png"); print("source | mask | edits:"); display(row.resize((min(W_,1400), int((S+40)*min(W_,1400)/W_))))


## 6 · Quantitative — structure kept, background kept, target changed

The paper's own PIE-Bench axes: **Canny SSIM** (structure), **BG SSIM / BG PSNR** (non-edited region), **CLIP** (the edit landed). Reference points from the paper: ConsistEdit reports Canny SSIM 0.8811, BG PSNR 36.76, BG SSIM 0.9869 — far above SDEdit / UniEdit-Flow / DiTCtrl.

In [ ]:
import numpy as np, torch
from PIL import Image
from skimage.metrics import structural_similarity as ssim
from transformers import CLIPModel, CLIPProcessor

keep = (np.asarray(mask.resize((1024,1024))) < 128)          # True = unedited region
print(f"background fraction: {keep.mean():.2%}")
src_np = np.asarray(src).astype(np.float32)/255.0
src_g = np.asarray(src.convert("L").resize((512,512)))

def canny(im):
    from skimage.feature import canny as _canny
    return _canny(np.asarray(im.convert("L").resize((512,512))), sigma=2.0)

def bg_scores(im):
    a = np.asarray(im.resize((1024,1024))).astype(np.float32)/255.0
    l1 = float(np.abs(a - src_np)[keep].mean())
    ss = float(ssim((a*keep[...,None]).astype(np.float32), (src_np*keep[...,None]).astype(np.float32),
                    channel_axis=2, data_range=1.0))
    mse = float((((a - src_np)*keep[...,None])**2).mean())
    return l1, ss, 10.0*np.log10(1.0/max(mse, 1e-12))

def c_ssim(im, ref_canny=None):
    ref = src_canny if ref_canny is None else ref_canny
    return float(ssim(canny(im)>0, ref>0, data_range=1.0))
src_canny = canny(src)

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEV).eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
def sim(img, text):
    b = proc(text=[text], images=[img], return_tensors="pt", padding=True).to(DEV)
    with torch.no_grad(): o = clip(**b)
    ie = o.image_embeds/o.image_embeds.norm(dim=-1,keepdim=True); te = o.text_embeds/o.text_embeds.norm(dim=-1,keepdim=True)
    return float((ie@te.T)[0,0])

print("edit | Canny SSIM (structure) | BG L1 / SSIM / PSNR (kept) | CLIP edit-direction")
ok = True
for sp, tp in EDITS:
    e = Image.open(f"edit_{tp.replace(' ','_')}.png")
    cs = c_ssim(e); l1, ss, psnr = bg_scores(e)
    se, ss0 = sim(e, tp), sim(src, tp)
    print(f"  {tp:>16}:  canny={cs:.3f}  |  L1={l1:.4f} SSIM={ss:.3f} PSNR={psnr:.1f}  |  "
          f"edit={se:.3f} vs source={ss0:.3f} -> {'PASS' if se>ss0 else 'REVIEW'}")
    ok = ok and (se > ss0) and (ss > 0.85) and (cs > 0.5)
print("\nstructure + background preservation + edit-direction:", "PASS" if ok else "REVIEW")


## 7 · The α sweep — progressive structural consistency

The capability that separates ConsistEdit from FlowEdit / KV-Edit: `consistency_strength` should move **structure** (Canny SSIM ↑) monotonically while the **edit still lands** (CLIP edit-direction stays positive). The paper's texture/structure disentanglement, as a curve.

In [ ]:
import torch, numpy as np
from PIL import Image
sp, tp = EDITS[1]        # the subject swap — the shape-changing case α is meant to govern
print("alpha | Canny SSIM (structure) | BG SSIM (kept) | CLIP(edit) - CLIP(source)")
rows = []
for a in [0.0, 0.3, 0.6, 1.0]:
    g = torch.Generator(DEV).manual_seed(0)
    im = pipe(image="src.png", mask="mask.png", source_prompt=sp, prompt=tp,
              height=1024, width=1024, T_steps=28, consistency_strength=a, generator=g).images[0]
    im.save(f"sweep_{a}.png")
    cs, (_, ss, _) = c_ssim(im), bg_scores(im)
    dr = sim(im, tp) - sim(src, tp)
    rows.append((a, cs, ss, dr)); print(f"  {a:.1f}  |  {cs:.3f}  |  {ss:.3f}  |  {dr:+.3f}")
mono = all(rows[i][1] <= rows[i+1][1] + 0.01 for i in range(len(rows)-1))
still_edits = all(r[3] > 0 for r in rows)
print(f"\nstructure rises with alpha: {'PASS' if mono else 'REVIEW'}; the edit lands at every alpha: "
      f"{'PASS' if still_edits else 'REVIEW'}")


## 8 · Head-to-head vs FlowEdit and KV-Edit

Same source, same edit, same kept region. FlowEdit re-synthesizes the background; KV-Edit holds it pixel-precise but has no structure control *inside* the edit region. ConsistEdit should sit at or above KV-Edit on background SSIM **and** beat both on Canny SSIM inside the edit region.

In [ ]:
import gc, torch, numpy as np
del pipe; gc.collect(); torch.cuda.empty_cache()   # free FLUX before loading another block

from diffusers import ModularPipeline
from skimage.feature import canny as _canny

def bg_and_canny(path_or_im):
    im = Image.open(path_or_im) if isinstance(path_or_im, str) else path_or_im
    _, ss, _ = bg_scores(im)
    return ss, c_ssim(im)

sp, tp = EDITS[1]
ce_ss, ce_cs = bg_and_canny(f"edit_{tp.replace(' ','_')}.png")
print(f"ConsistEdit   :  BG SSIM={ce_ss:.3f}  Canny SSIM={ce_cs:.3f}")

fe = ModularPipeline.from_pretrained("remyxai/flowedit-flux-modular", trust_remote_code=True)
assert type(fe.blocks).__name__ == "FlowEditBlock"
fe.load_components(dtype=DT); fe.to(DEV)
g = torch.Generator(DEV).manual_seed(0)
fim = fe(image="src.png", source_prompt=sp, prompt=tp, height=1024, width=1024,
         T_steps=28, src_guidance_scale=1.5, tar_guidance_scale=5.5, n_max=24, n_min=0,
         generator=g).images[0]
fe_ss, fe_cs = bg_and_canny(fim); print(f"FlowEdit     :  BG SSIM={fe_ss:.3f}  Canny SSIM={fe_cs:.3f}")
del fe; gc.collect(); torch.cuda.empty_cache()

kv = ModularPipeline.from_pretrained("remyxai/kv-edit-flux-modular", trust_remote_code=True)
assert type(kv.blocks).__name__ == "KVEditBlock"
kv.load_components(dtype=DT); kv.to(DEV)
g = torch.Generator(DEV).manual_seed(0)
kim = kv(image="src.png", mask="mask.png", source_prompt=sp, prompt=tp, height=1024, width=1024,
         T_steps=28, guidance_scale=3.5, src_guidance_scale=1.0, generator=g).images[0]
kv_ss, kv_cs = bg_and_canny(kim); print(f"KV-Edit      :  BG SSIM={kv_ss:.3f}  Canny SSIM={kv_cs:.3f}")
print(f"\nverdict -> background: ConsistEdit {'>=' if ce_ss >= kv_ss - 0.01 else '<'} KV-Edit, "
      f"{'>=' if ce_ss >= fe_ss else '<'} FlowEdit | structure: ConsistEdit "
      f"{'wins' if ce_cs >= max(fe_cs, kv_cs) else 'REVIEW'}")


## Verdict
PASS = `loaded block: ConsistEditBlock` + the edit lands (CLIP edit-direction > 0) + the kept region holds (BG SSIM > 0.85) + structure rises monotonically with `consistency_strength` + background at or above KV-Edit and above FlowEdit. On pass: human confirms, flip the repo public, add the Colab badge + the umbrella collection.